# Documentación del Proceso ETL: Consolidación Histórica y Enriquecimiento Espacial del Valor de Referencia

## 1. Fuentes de Datos e Ingesta

El proyecto se alimenta de tres fuentes oficiales abiertas provistas por el Distrito Capital a través del portal de Datos Abiertos Bogotá e IDECA:

Valor de Referencia por Metro Cuadrado de Terreno: Conjunto de datos históricos en formato comprimido (.zip) que agrupa los polígonos de manzanas catastrales y sus respectivos valores económicos de referencia calculados año tras año (2012–2026).

Fuente: https://datosabiertos.bogota.gov.co/dataset/valor-de-referencia-por-metro-cuadrado-de-terreno

Sector Catastral: Capa geográfica oficial (.zip / Shapefile) que define la división político-administrativa a nivel de barrios o sectores catastrales, utilizada para contextualizar espacialmente cada manzana.

Fuente: https://datosabiertos.bogota.gov.co/dataset/sector-catastral

Localidad (loca.gpkg): Capa geoespacial en formato GeoPackage que delimita las localidades del Distrito Capital, permitiendo establecer la jerarquía territorial superior.

Fuente: https://datosabiertos.bogota.gov.co/dataset/localidad-bogota-d-c

## 2. Metodología y Procesamiento (ETL Pipeline)

El pipeline automatizado implementa un proceso de Extracción, Transformación y Carga (ETL) espacial estructurado en los siguientes pasos:

Ingestión y Estandarización: Se descomprimen de forma masiva los archivos históricos anuales, extrayendo las geometrías de las manzanas y normalizando los nombres de columnas heterogéneas (mancodigo, v_ref, shape_area, shape_len, ano) para unificarlas en un único marco de datos temporal.

Enriquecimiento Espacial por Intersección (sjoin): Utilizando la librería geopandas, se reproyectan las capas bajo un mismo sistema de referencia de coordenadas (CRS) y se ejecutan cruces espaciales secuenciales basados en el predicado intersects. Esto permite asignar a cada manzana histórica su respectivo barrio (desde el Sector Catastral) y localidad (desde el archivo GeoPackage).

Deduplicación e Integridad Estructural: Para corregir la multiplicación de registros derivada de manzanas que intersectan bordes administrativos, se implementa una limpieza basada en el identificador único original (id_original), garantizando una única asignación espacial por manzana.

Depuración de Datos Incompletos: Se aplica un filtro de exclusión (dropna) para remover registros que carezcan de variables económicas o administrativas esenciales (barrio, localidad, v_ref, shape_area, shape_len).

## 3. Productos Finales Generados

Como resultado del procesamiento, se obtienen dos datasets estructurados:

valor_referencia_historico_tabular.csv: Archivo tabular optimizado para análisis estadísticos y econométricos. Contiene el histórico consolidado a nivel de manzana, con sus valores de referencia, dimensiones físicas y jerarquía administrativa (barrio y localidad).

valor_referencia_historico_espacial.gpkg: Archivo GeoPackage que conserva la geometría vectorial completa, permitiendo visualizaciones geográficas y análisis de autocorrelación espacial en software SIG (QGIS, ArcGIS) o bibliotecas de Python.

# Librerias

In [211]:
import os
import zipfile
import glob
import re
import numpy as np
import pandas as pd
import geopandas as gpd

# 0. Extracción de datos

In [212]:
# ==============================================================================
# PASO 0: EXTRACCIÓN AUTOMÁTICA DE ARCHIVOS COMPRIMIDOS
# ==============================================================================
os.makedirs('data_historica', exist_ok=True)
for f in glob.glob('*.zip'):
    if 'sector' not in f.lower() and 'transmilenio' not in f.lower():
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall(os.path.join('data_historica', os.path.splitext(f)[0]))

os.makedirs('temp_sector', exist_ok=True)
for f in glob.glob('*sector*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_sector')

os.makedirs('temp_transmilenio', exist_ok=True)
for f in glob.glob('*transmilenio*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_transmilenio')

print("--- EXTRACCIÓN COMPLETADA ---")

--- EXTRACCIÓN COMPLETADA ---


# 1. Unificación Localidad - Barrio - Geometria

In [213]:
# ==============================================================================
# PASO 1: DICCIONARIO ESPACIAL BASE (LOCALIDAD - BARRIO - GEOMETRÍA)
# ==============================================================================
archivo_sector = glob.glob('temp_sector/**/*.shp', recursive=True)[0]
gdf_sector = gpd.read_file(archivo_sector)

gdf_localidad = gpd.read_file('loca.gpkg')
gdf_localidad = gdf_localidad.to_crs(gdf_sector.crs)

gdf_sector = gdf_sector.rename(columns={'SCANOMBRE': 'barrio'})

gdf_barrios_base = gpd.sjoin(gdf_sector[['barrio', 'geometry']], gdf_localidad[['LocNombre', 'geometry']], how='left', predicate='intersects')
gdf_barrios_base = gdf_barrios_base.rename(columns={'LocNombre': 'localidad'})
gdf_barrios_base = gdf_barrios_base.drop(columns=['index_right'])

# Deduplicar polígonos de barrio
gdf_barrios_base = gdf_barrios_base.drop_duplicates(subset=['barrio']).reset_index(drop=True)

print(f"\n[PASO 1] Tamaño del Diccionario Espacial Base (Barrios): {gdf_barrios_base.shape[0]} filas, {gdf_barrios_base.shape[1]} columnas")


[PASO 1] Tamaño del Diccionario Espacial Base (Barrios): 1150 filas, 3 columnas


In [214]:
gdf_barrios_base.head(5)

,barrio,geometry,localidad
0,EL BOSQUE,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525...",USME
1,BILBAO,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ...",SUBA
2,CIUDAD KENNEDY CENTRAL,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657...",KENNEDY
3,CONCEPCION NORTE,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727...",CHAPINERO
4,COLON,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043...",PUENTE ARANDA


# 2. Consolidación valores historicos de metro cuadrado

In [215]:
# ==============================================================================
# PASO 2: CONSOLIDACIÓN DE VALORES DE REFERENCIA HISTÓRICOS (VHI)
# ==============================================================================
lista_gdfs = []
target_crs = "EPSG:4326"
 
for shp in glob.glob('data_historica/**/*.shp', recursive=True):
    gdf = gpd.read_file(shp)
   
    # Asegurar que todas las capas usen el mismo CRS para poder concatenar
    if gdf.crs is None:
        gdf.set_crs(target_crs, inplace=True)
    elif gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
       
    match = re.search(r'20\d{2}', shp)
    ano = int(match.group(0)) if match else None
    cols = {c.lower(): c for c in gdf.columns}
    datos = {
        'mancodigo': gdf[cols.get('mancodigo')] if cols.get('mancodigo') else None,
        'v_ref': gdf[cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')] if (cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')) else None,
        'ano': ano,
        'geometry': gdf['geometry']
    }
    lista_gdfs.append(gpd.GeoDataFrame(datos, crs=target_crs))
 
gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)
print(f"[PASO 2a] Tamaño de Registros de Manzanas Históricas (gdf_maestro): {gdf_maestro.shape[0]} filas, {gdf_maestro.shape[1]} columnas")
 
gdf_barrios_base = gdf_barrios_base.to_crs(gdf_maestro.crs)
 
# Cruce espacial de manzanas con diccionario de barrios
gdf_historico = gpd.sjoin(gdf_maestro, gdf_barrios_base[['barrio', 'localidad', 'geometry']], how='inner', predicate='intersects')
if 'index_right' in gdf_historico.columns:
    gdf_historico = gdf_historico.drop(columns=['index_right'])
 
# Mediana del valor de referencia por barrio y año
df_vhi_barrio = gdf_historico.groupby(['localidad', 'barrio', 'ano'])['v_ref'].median().reset_index()
print(f"[PASO 2b] Tamaño del Consolidado VHI por Barrio y Año: {df_vhi_barrio.shape[0]} filas, {df_vhi_barrio.shape[1]} columnas")

[PASO 2a] Tamaño de Registros de Manzanas Históricas (gdf_maestro): 518935 filas, 4 columnas
[PASO 2b] Tamaño del Consolidado VHI por Barrio y Año: 13147 filas, 4 columnas


In [216]:
df_vhi_barrio.head(5)

,localidad,barrio,ano,v_ref
0,ANTONIO NARIÑO,CARACAS,2015,1300000.0
1,ANTONIO NARIÑO,CARACAS,2016,1400000.0
2,ANTONIO NARIÑO,CARACAS,2017,1700000.0
3,ANTONIO NARIÑO,CARACAS,2018,1700000.0
4,ANTONIO NARIÑO,CARACAS,2019,1750000.0


# 3. Extracción datos transmilenio

In [217]:
# ==============================================================================
# PASO 3: DENSIDAD DE INFRAESTRUCTURA DE TRANSMILENIO
# ==============================================================================
archivo_transmilenio = glob.glob('temp_transmilenio/**/*.shp', recursive=True)[0]
gdf_transmilenio = gpd.read_file(archivo_transmilenio)
print(f"[PASO 3a] Estaciones de Transmilenio leídas: {gdf_transmilenio.shape[0]} filas, {gdf_transmilenio.shape[1]} columnas")

if gdf_barrios_base.crs != gdf_transmilenio.crs:
    gdf_transmilenio = gdf_transmilenio.to_crs(gdf_barrios_base.crs)

# Conteo espacial de estaciones por barrio
estaciones_por_barrio = gpd.sjoin(gdf_transmilenio, gdf_barrios_base[['barrio', 'geometry']], how='inner', predicate='intersects')
conteo_estaciones = estaciones_por_barrio.groupby('barrio').size().reset_index(name='num_estaciones_transmilenio')
print(f"[PASO 3b] Barrios con al menos una estación de Transmilenio: {conteo_estaciones.shape[0]} barrios")

[PASO 3a] Estaciones de Transmilenio leídas: 153 filas, 28 columnas
[PASO 3b] Barrios con al menos una estación de Transmilenio: 111 barrios


In [218]:
conteo_estaciones

,barrio,num_estaciones_transmilenio
0,ABRAHAM LINCOLN,1
1,ACEVEDO TEJADA,2
2,ALTOS DE CHOZICA,1
3,AUTOPISTA MUZU,1
4,AUTOPISTA MUZU ORIENTAL,1
...,...,...
106,SIMON BOLIVAR,1
107,SOCIEGO,1
108,SOSIEGO SUR,1
109,SURAMERICA,1


# 4. Cálculo CAGR 2015-2025

In [219]:
# ==============================================================================
# PASO 4: CÁLCULO DE LA TASA DE CRECIMIENTO ANUAL COMPUESTA (CAGR 2015-2025)
# ==============================================================================
df_cagr_base = df_vhi_barrio[df_vhi_barrio['ano'].isin([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])]
df_pivot = df_cagr_base.pivot(index=['localidad', 'barrio'], columns='ano', values='v_ref').reset_index()
df_pivot = df_pivot.rename(columns={2015: 'v_ref_2015', 2016: 'v_ref_2016', 2017: 'v_ref_2017', 
                                    2018: 'v_ref_2018', 2019: 'v_ref_2019', 2020: 'v_ref_2020', 
                                    2021: 'v_ref_2021', 2022: 'v_ref_2022', 2023: 'v_ref_2023', 
                                    2024: 'v_ref_2024', 2025: 'v_ref_2025', 2026: 'v_ref_2026'})

In [220]:
# ==============================================================================
# FUNCIÓN DE AUDITORÍA DE ANOMALÍAS (NULOS Y CEROS)
# ==============================================================================
def auditar_datos_inmobiliarios(df, etapa="ANTES DE LA IMPUTACIÓN"):
    """
    Función para auditar el estado de los valores de referencia (2015, 2025, 2026).
    Muestra conteo de nulos, registros afectados y desglose específico de anomalías.
    """
    print(f"\n================ CLINICAL AUDIT: {etapa} ================")
    
    cols_eval = ['v_ref_2015', 'v_ref_2025', 'v_ref_2026']
    
    # 1. Conteo total de valores nulos (NaN) actuales por columna
    print("\n1. Conteo de valores nulos (NaN) por columna:")
    print(df[cols_eval].isnull().sum())
    
    # 2. Conteo de ceros (0) que representan datos faltantes disfrazados
    print("\n2. Conteo de valores en cero (0) por columna:")
    print((df[cols_eval] == 0).sum())
    
    # 3. Conteo específico por año de anomalías (NaN o 0)
    print("\n3. Desglose detallado de anomalías (NaN o 0) por año:")
    for col in cols_eval:
        n_nans = df[col].isnull().sum()
        n_zeros = (df[col] == 0).sum()
        print(f"   - {col}: {n_nans} NaNs | {n_zeros} ceros (Total anomalías: {n_nans + n_zeros})")
    
    # 4. Filtrar filas con anomalías en el rango clave del CAGR (2015 a 2025) y 2026
    condicion_critica = (
        df['v_ref_2015'].isnull() | (df['v_ref_2015'] == 0) |
        df['v_ref_2025'].isnull() | (df['v_ref_2025'] == 0) |
        df['v_ref_2026'].isnull() | (df['v_ref_2026'] == 0) 
    )
    filas_criticas = df[condicion_critica]
    print(f"\n4. Total de registros afectados con anomalías entre 2015 - 2025 y 2026: {len(filas_criticas)}")
    
    # 5. Mostrar una muestra de los registros afectados si los hay
    if len(filas_criticas) > 0:
        print("\n5. Muestra de registros con anomalías críticas:")
        display(filas_criticas[['localidad', 'barrio', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026']].head(10))
    else:
        print("\n5. ¡No se encontraron registros críticos entre 2015 - 2025 y 2026!")
    print("================================================================")

In [221]:
auditar_datos_inmobiliarios(df_pivot, etapa="ESTADO INICIAL (CON CEROS Y NULOS)")


================ CLINICAL AUDIT: ESTADO INICIAL (CON CEROS Y NULOS) ================

1. Conteo de valores nulos (NaN) por columna:
ano
v_ref_2015     0
v_ref_2025     4
v_ref_2026    13
dtype: int64

2. Conteo de valores en cero (0) por columna:
ano
v_ref_2015    68
v_ref_2025    36
v_ref_2026     0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 68 ceros (Total anomalías: 68)
   - v_ref_2025: 4 NaNs | 36 ceros (Total anomalías: 40)
   - v_ref_2026: 13 NaNs | 0 ceros (Total anomalías: 13)

4. Total de registros afectados con anomalías entre 2015 - 2025 y 2026: 92

5. Muestra de registros con anomalías críticas:


ano,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026
35,BOSA,BOSA 37,0.0,81000.0,1391520.0
40,BOSA,CAMPO VERDE,0.0,1431294.5,1498952.0
42,BOSA,CANAVERALEJO RURAL,0.0,33600.0,38100.0
45,BOSA,CAÑAVERALEJO I,0.0,33600.0,38100.0
56,BOSA,EL CORZO II,0.0,33600.0,38100.0
60,BOSA,EL REMANSO,0.0,130100.0,162600.0
88,BOSA,OSORIO XXIII,302260.0,0.0,541169.0
97,BOSA,SAN BERNARDINO XIX,0.0,1155000.0,1525000.0
98,BOSA,SAN BERNARDINO XVIII,0.0,510000.0,162100.0
99,BOSA,SAN BERNARDINO XXII URBANO,0.0,463486.0,821500.0


In [222]:
anios_historicos = [f'v_ref_{anio}' for anio in range(2015, 2026)]  # 2015 a 2025
col_objetivo = ['v_ref_2026']

# A. Normalización de ceros a NaN: 
# Los ceros en precios inmobiliarios por metro cuadrado corresponden a vacíos de información,
# no a valor comercial real, por lo que se estandarizan a NaN para evitar sesgos matemáticos.
cols_a_limpiar = anios_historicos + col_objetivo
df_pivot[cols_a_limpiar] = df_pivot[cols_a_limpiar].replace(0, np.nan)

# B. Imputación 2015 (Corte Base):
# Se aplica la mediana del valor de referencia agrupada por localidad. 
# Justificación: Preserva la tendencia macroespacial zonal sin distorsionar con valores extremos (outliers).
df_pivot['v_ref_2015'] = df_pivot['v_ref_2015'].fillna(
    df_pivot.groupby('localidad')['v_ref_2015'].transform('median')
)

# C. Imputación 2025 (Trayectoria Histórica + Respaldo Zonal):
# 1. Primero se intenta rellenar usando interpolación lineal con los años intermedios disponibles por cada fila,
#    respetando la inercia de precios particular de cada barrio.
df_pivot['v_ref_2025'] = df_pivot['v_ref_2025'].fillna(
    df_pivot[anios_historicos].interpolate(method='linear', axis=1)['v_ref_2025']
)
# 2. Si aún persisten vacíos (barrios sin ningún histórico intermedio), se aplica la mediana de la localidad como respaldo.
df_pivot['v_ref_2025'] = df_pivot['v_ref_2025'].fillna(
    df_pivot.groupby('localidad')['v_ref_2025'].transform('median')
)

# D. Tratamiento de 2026 (Año Objetivo / Predicción):
# Se dejan intencionalmente como NaN. Al ser el objetivo que evaluará el modelo de Machine Learning,
# no se imputan en el set de entrenamiento/prueba para evitar contaminar las métricas de error (MSE/RMSE).

In [223]:
auditar_datos_inmobiliarios(df_pivot, etapa="POST-IMPUTACIÓN (RESULTADO FINAL)")


================ CLINICAL AUDIT: POST-IMPUTACIÓN (RESULTADO FINAL) ================

1. Conteo de valores nulos (NaN) por columna:
ano
v_ref_2015     5
v_ref_2025     0
v_ref_2026    13
dtype: int64

2. Conteo de valores en cero (0) por columna:
ano
v_ref_2015    0
v_ref_2025    0
v_ref_2026    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 5 NaNs | 0 ceros (Total anomalías: 5)
   - v_ref_2025: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 13 NaNs | 0 ceros (Total anomalías: 13)

4. Total de registros afectados con anomalías entre 2015 - 2025 y 2026: 13

5. Muestra de registros con anomalías críticas:


ano,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026
207,CIUDAD BOLIVAR,EL MOCHUELO III RURAL,27996.0,55498.5,NaN
287,ENGATIVA,CAMPO EUCARISTICO,80000.0,170930.0,NaN
382,FONTIBON,EL PANTANO,92344.5,34450.0,NaN
467,KENNEDY,OSORIO,75000.0,111000.0,NaN
471,KENNEDY,OSORIO XII RURAL,75000.0,111000.0,NaN
838,SUMAPAZ,LA UNION URBANO,NaN,39390.0,NaN
839,SUMAPAZ,NAZARETH,NaN,35000.0,NaN
840,SUMAPAZ,NAZARETH URBANO,NaN,40000.0,NaN
841,SUMAPAZ,SAN JUAN,NaN,37500.0,NaN
842,SUMAPAZ,SAN JUAN URBANO,NaN,37500.0,NaN


In [224]:
# 1. Filtrar el DataFrame para eliminar aquellas filas que siguen teniendo 
# anomalías críticas (NaN o ceros) en 2015 o 2025 después de la imputación.
# (Esto removerá exactamente esos 5 registros que no pudieron ser imputados).
df_pivot = df_pivot[
    ~(
        df_pivot['v_ref_2015'].isnull() | (df_pivot['v_ref_2015'] == 0) |
        df_pivot['v_ref_2025'].isnull() | (df_pivot['v_ref_2025'] == 0)
    )
].copy()

print(f"🧹 Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: {df_pivot.shape}")

# 2. Volver a correr la función de auditoría para verificar el estado final
auditar_datos_inmobiliarios(df_pivot, etapa="ESTADO FINAL (TRAS FILTRAR HUÉRFANOS)")

🧹 Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: (1098, 14)

================ CLINICAL AUDIT: ESTADO FINAL (TRAS FILTRAR HUÉRFANOS) ================

1. Conteo de valores nulos (NaN) por columna:
ano
v_ref_2015    0
v_ref_2025    0
v_ref_2026    8
dtype: int64

2. Conteo de valores en cero (0) por columna:
ano
v_ref_2015    0
v_ref_2025    0
v_ref_2026    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2025: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 8 NaNs | 0 ceros (Total anomalías: 8)

4. Total de registros afectados con anomalías entre 2015 - 2025 y 2026: 8

5. Muestra de registros con anomalías críticas:


ano,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026
207,CIUDAD BOLIVAR,EL MOCHUELO III RURAL,27996.0,55498.50,NaN
287,ENGATIVA,CAMPO EUCARISTICO,80000.0,170930.00,NaN
382,FONTIBON,EL PANTANO,92344.5,34450.00,NaN
467,KENNEDY,OSORIO,75000.0,111000.00,NaN
471,KENNEDY,OSORIO XII RURAL,75000.0,111000.00,NaN
867,TEUSAQUILLO,PARQUE DISTRITAL SALITRE,80000.0,139363.69,NaN
927,USAQUEN,MIRADOR DEL NORTE,80000.0,138000.00,NaN
1006,USME,EL MOCHUELO ALTO RURAL,27996.0,69364.00,NaN


In [225]:
# Calcular CAGR de 11 años (2015 a 2025)
n_anos = 2025 - 2015
df_pivot['cagr_15_25'] = (df_pivot['v_ref_2025'] / df_pivot['v_ref_2015']) ** (1 / n_anos) - 1
print(f"[PASO 4] Tamaño de la tabla de CAGR (2015-2025): {df_pivot.shape[0]} filas, {df_pivot.shape[1]} columnas")

[PASO 4] Tamaño de la tabla de CAGR (2015-2025): 1098 filas, 15 columnas


In [226]:
df_pivot 

ano,localidad,barrio,v_ref_2015,v_ref_2016,v_ref_2017,v_ref_2018,v_ref_2019,v_ref_2020,v_ref_2021,v_ref_2022,v_ref_2023,v_ref_2024,v_ref_2025,v_ref_2026,cagr_15_25
0,ANTONIO NARIÑO,CARACAS,1300000.00,1.400000e+06,1700000.00,1700000.00,1750000.0,1900000.00,2200000.0,2200000.0,2500000.0,2570000.0,2679000.0,3200000.0,0.074986
1,ANTONIO NARIÑO,EDUARDO SANTOS,900000.00,1.100000e+06,1200000.00,1600000.00,1800000.0,2000000.00,2350000.0,2350000.0,2560000.0,2800000.0,2950000.0,3000000.0,0.126051
2,ANTONIO NARIÑO,EL VERGEL,912500.00,1.037500e+06,1250000.00,1625000.00,1800000.0,2000000.00,2350000.0,2350000.0,2560000.0,2750000.0,2900000.0,3000000.0,0.122578
3,ANTONIO NARIÑO,LA FRAGUITA,1000000.00,1.200000e+06,1350000.00,1600000.00,1800000.0,2000000.00,2350000.0,2350000.0,2560000.0,2750000.0,2900000.0,3000000.0,0.112346
4,ANTONIO NARIÑO,SANTANDER,1000000.00,1.200000e+06,1250000.00,1600000.00,2000000.0,2100000.00,2350000.0,2350000.0,2650000.0,2820000.0,3040000.0,3150000.0,0.117602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1098,USME,VILLA ISRAEL,200000.00,2.200000e+05,240000.00,400000.00,500000.0,550000.00,650000.0,650000.0,700000.0,750000.0,800000.0,295200.0,0.148698
1099,USME,VILLABEL,135000.00,1.500000e+05,165000.00,187036.50,199600.5,164305.35,249000.0,340963.0,319640.0,335961.0,203000.0,393034.0,0.041637
1100,USME,YOMASA,230000.00,2.300000e+05,230000.00,300000.00,340000.0,360000.00,395000.0,395000.0,430000.0,440000.0,450000.0,480000.0,0.069420
1101,USME,YOMASA NORTE,34001.50,3.400158e+04,70095.50,42500.00,42350.0,42350.00,NaN,NaN,138551.0,138546.0,170694.0,170124.0,0.175092


# 5. Cargar listings agrupados

In [227]:
listings = pd.read_csv("listings_agrupado_barrio_localidad.csv")
len(listings)

654

In [228]:
listings = listings.rename(columns={'conteo': 'num_listings_airbnb'})

In [229]:
listings

,localidad,barrio,num_listings_airbnb
0,ANTONIO NARIÑO,CIUDAD BERNA,1
1,ANTONIO NARIÑO,CIUDAD JARDIN SUR,4
2,ANTONIO NARIÑO,EDUARDO FREY,5
3,ANTONIO NARIÑO,LA FRAGUA,1
4,ANTONIO NARIÑO,LA FRAGUITA,7
...,...,...,...
649,USME,LA FISCALA NORTE,1
650,USME,LA REGADERA,1
651,USME,MONTEBLANCO,1
652,USME,OLARTE,3


# 6. Unificación datos

In [230]:
# ==============================================================================
# PASO 5: UNIFICACIÓN TOTAL, ORDENAMIENTO Y EXPORTACIÓN
# ==============================================================================
matriz_final = gdf_barrios_base.copy()

# Fusión con CAGR y valores de referencia
# CORRECCIÓN: Se agregó 'localidad' a la lista de columnas seleccionadas de df_pivot
matriz_final = matriz_final.merge(df_pivot[['barrio', 'localidad', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25']], on=['barrio', 'localidad'], how='left')

# Fusión con Transmilenio
matriz_final = matriz_final.merge(conteo_estaciones, on=['barrio'], how='left')
matriz_final['num_estaciones_transmilenio'] = matriz_final['num_estaciones_transmilenio'].fillna(0).astype(int)

# Cruzamos la información manteniendo todos los barrios del dataset general
matriz_final = matriz_final.merge(listings, on=['barrio', 'localidad'], how='left')

# Rellenamos con 0 los valores de los barrios que no estaban en listings
matriz_final['num_listings_airbnb'] = matriz_final['num_listings_airbnb'].fillna(0)

# Manejo de nulos en CAGR si un barrio no tiene serie completa
matriz_final['cagr_15_25'] = matriz_final['cagr_15_25'].fillna(0)

In [231]:
matriz_final = matriz_final[
    ~(
        matriz_final['v_ref_2015'].isnull() | (matriz_final['v_ref_2015'] == 0) |
        matriz_final['v_ref_2025'].isnull() | (matriz_final['v_ref_2025'] == 0)
    )
].copy()

print(f"🧹 Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: {matriz_final.shape}")

auditar_datos_inmobiliarios(matriz_final, etapa="ESTADO FINAL (TRAS UNIÓN)")

🧹 Se han eliminado los registros huérfanos. Nuevas dimensiones del DataFrame: (1098, 9)

================ CLINICAL AUDIT: ESTADO FINAL (TRAS UNIÓN) ================

1. Conteo de valores nulos (NaN) por columna:
v_ref_2015    0
v_ref_2025    0
v_ref_2026    8
dtype: int64

2. Conteo de valores en cero (0) por columna:
v_ref_2015    0
v_ref_2025    0
v_ref_2026    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2025: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 8 NaNs | 0 ceros (Total anomalías: 8)

4. Total de registros afectados con anomalías entre 2015 - 2025 y 2026: 8

5. Muestra de registros con anomalías críticas:


,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026
87,FONTIBON,EL PANTANO,92344.5,34450.00,NaN
174,TEUSAQUILLO,PARQUE DISTRITAL SALITRE,80000.0,139363.69,NaN
526,CIUDAD BOLIVAR,EL MOCHUELO III RURAL,27996.0,55498.50,NaN
598,KENNEDY,OSORIO XII RURAL,75000.0,111000.00,NaN
656,USME,EL MOCHUELO ALTO RURAL,27996.0,69364.00,NaN
742,KENNEDY,OSORIO,75000.0,111000.00,NaN
919,ENGATIVA,CAMPO EUCARISTICO,80000.0,170930.00,NaN
1036,USAQUEN,MIRADOR DEL NORTE,80000.0,138000.00,NaN


In [232]:
matriz_final.describe()

,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb
count,1.098000e+03,1.098000e+03,1.090000e+03,1098.000000,1098.000000,1098.000000
mean,9.670187e+05,2.181089e+06,2.462160e+06,0.090624,0.130237,12.868852
std,8.222106e+05,1.595993e+06,1.764579e+06,0.092669,0.420938,51.805744
min,6.000000e+03,6.700000e+03,1.260000e+04,-0.406310,0.000000,0.000000
25%,3.600000e+05,8.211108e+05,9.500000e+05,0.074885,0.000000,0.000000
50%,8.500000e+05,2.150000e+06,2.450000e+06,0.095297,0.000000,0.000000
75%,1.300000e+06,3.150000e+06,3.456000e+06,0.119773,0.000000,5.000000
max,6.450000e+06,1.352500e+07,1.778500e+07,0.509304,3.000000,883.000000


In [233]:
# ---> ORDENAR LAS COLUMNAS SEGÚN LO SOLICITADO <---
columnas_ordenadas = [
    'localidad',
    'barrio',
    'v_ref_2015',
    'v_ref_2025',
    'v_ref_2026',
    'cagr_15_25',
    'num_estaciones_transmilenio',
    'num_listings_airbnb'
] + ['geometry']

matriz_final = matriz_final[columnas_ordenadas]

# Asegurar que se mantenga como GeoDataFrame espacial
matriz_final = gpd.GeoDataFrame(matriz_final, geometry='geometry')

# Exportación de archivos finales ordenados
matriz_final.to_csv('matriz_barrios_unificada.csv', index=False)
matriz_final.to_file('matriz_barrios_unificada.gpkg', driver='GPKG')

print("\n¡Matriz unificada, ordenada y exportada con éxito!")
print(f"Columnas finales actuales: {matriz_final.columns.tolist()}")


¡Matriz unificada, ordenada y exportada con éxito!
Columnas finales actuales: ['localidad', 'barrio', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25', 'num_estaciones_transmilenio', 'num_listings_airbnb', 'geometry']


In [234]:
matriz_final

,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry
0,USME,EL BOSQUE,290000.0,650000.0,660000.0,0.084056,0,0.0,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525..."
1,SUBA,BILBAO,750000.0,1790000.0,2050000.0,0.090886,0,0.0,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657..."
3,CHAPINERO,CONCEPCION NORTE,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727..."
4,PUENTE ARANDA,COLON,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043..."
...,...,...,...,...,...,...,...,...,...
1145,CIUDAD BOLIVAR,MEISSEN,64000.0,1198158.0,1395000.0,0.340397,0,1.0,"POLYGON ((-74.13463 4.56098, -74.13427 4.56086..."
1146,RAFAEL URIBE URIBE,LOS MOLINOS,450000.0,2750000.0,2600000.0,0.198428,0,0.0,"POLYGON ((-74.10753 4.55803, -74.10742 4.55802..."
1147,SUBA,EL PLAN,1400000.0,3400000.0,3798483.0,0.092786,0,38.0,"POLYGON ((-74.06395 4.74181, -74.06391 4.74181..."
1148,CIUDAD BOLIVAR,ISMAEL PERDOMO,400000.0,1040000.0,1050000.0,0.100265,0,0.0,"POLYGON ((-74.16717 4.58155, -74.16718 4.58155..."
